<a href="https://colab.research.google.com/github/Nurdaylight/A-Karpathy-repl/blob/main/Text_scraper_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip -q install playwright
!python -m playwright install chromium
!python -m playwright install-deps chromium


In [28]:
from bs4 import BeautifulSoup

# read file containing HTML
with open("novelbin_chapters/input.txt", "r", encoding="utf-8") as f:
    html = f.read()

soup = BeautifulSoup(html, "html.parser")

# extract all links
links = [a["href"] for a in soup.find_all("a", href=True)]


In [ ]:
import asyncio
import os
from playwright.async_api import async_playwright

def clean_lines(text: str) -> str:
    lines = [ln.strip() for ln in text.splitlines()]
    return "\n".join([ln for ln in lines if ln])

async def scrape_chapters_from_links(links, delay_sec: float = 1.0):
    os.makedirs("novelbin_chapters", exist_ok=True)

    async with async_playwright() as p:
        browser = await p.chromium.launch(
            headless=True,
            args=[
                "--no-sandbox",
                "--disable-setuid-sandbox",
                "--disable-dev-shm-usage",
                "--disable-gpu",
                "--no-zygote",
                "--single-process",
            ],
        )

        context = await browser.new_context(
            user_agent="Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/122.0.0.0 Safari/537.36",
            locale="en-US",
        )

        page = await context.new_page()
        await page.set_extra_http_headers({
            "Accept-Language": "en-US,en;q=0.9",
            "Referer": "https://novelbin.com/",
        })

        for idx, url in enumerate(links):
            print(f"\nScraping ({idx+1}/{len(links)}): {url}")

            await page.goto(url, wait_until="domcontentloaded", timeout=6000)
            await page.wait_for_selector("#chr-content", timeout=3000)

            title = (await page.locator("h2").first.inner_text()).strip()
            raw = await page.locator("#chr-content").inner_text()

            text = clean_lines(raw)

            filename = f"reading/chapter_{links[idx][57:68]}.txt"
            with open(filename, "w", encoding="utf-8") as f:
                f.write(title + "\n\n" + text)

            print(f"Saved → {filename}")
            break

        await context.close()
        await browser.close()
for i in range(50):
  await scrape_chapters_from_links(links[i+200:i+201], delay_sec=10.164)
  await asyncio.sleep(3)


Scraping (1/1): https://novelbin.com/b/return-of-the-runebound-professor/chapter-200-panic
Saved → reading/chapter_chapter-200.txt

Scraping (1/1): https://novelbin.com/b/return-of-the-runebound-professor/chapter-201-life
Saved → reading/chapter_chapter-201.txt

Scraping (1/1): https://novelbin.com/b/return-of-the-runebound-professor/chapter-202-that-sucks
Saved → reading/chapter_chapter-202.txt

Scraping (1/1): https://novelbin.com/b/return-of-the-runebound-professor/chapter-203-meld
Saved → reading/chapter_chapter-203.txt

Scraping (1/1): https://novelbin.com/b/return-of-the-runebound-professor/chapter-204-questions
Saved → reading/chapter_chapter-204.txt


In [58]:
links[400][57:68]

'chapter-397'

In [64]:
import shutil
import os

folders_to_delete = ["novelbin_chapters", "hhhh"]

for folder in folders_to_delete:
    if os.path.exists(folder):
        shutil.rmtree(folder)
        print(f"Deleted folder: {folder}")
    else:
        print(f"Folder not found: {folder}")

Folder not found: novelbin_chapters
Folder not found: hhhh
